In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

users_df = pd.read_csv("../data/users.csv")
events_df = pd.read_csv("../data/events.csv", parse_dates=["timestamp"])

print(f"Users: {len(users_df):,}")
print(f"Events: {len(events_df):,}")
print(f"\nEvents columns: {list(events_df.columns)}")

Users: 1,000
Events: 2,766,533

Events columns: ['user_id', 'is_bot', 'video_id', 'event_type', 'timestamp', 'ip_address', 'session_duration_sec']


### velocity of features

In [3]:
# Group all events by user and calculate behavioral summaries
velocity = events_df.groupby("user_id").agg(
    total_events        = ("event_type", "count"),
    unique_videos       = ("video_id", "nunique"),
    unique_ips          = ("ip_address", "nunique"),
    avg_session_duration= ("session_duration_sec", "mean"),
    total_days_active   = ("timestamp", lambda x: x.dt.date.nunique()),
).reset_index()

velocity["events_per_day"] = velocity["total_events"] / velocity["total_days_active"]

print(velocity.head(3))
print(f"\nShape: {velocity.shape}")

     user_id  total_events  unique_videos  unique_ips  avg_session_duration  \
0  BOT_00000         10822             20           1              5.544077   
1  BOT_00001         10313             20           1              5.473480   
2  BOT_00002         10765             20           1              5.501254   

   total_days_active  events_per_day  
0                 30      360.733333  
1                 30      343.766667  
2                 30      358.833333  

Shape: (1000, 7)


In [4]:
# Calculate what fraction of each user's actions are likes, views, comments etc.
event_counts = events_df.groupby(["user_id", "event_type"]).size().unstack(fill_value=0)



In [ ]:
# Convert raw counts to ratios (out of total events)
event_ratios = event_counts.div(event_counts.sum(axis=1), axis=0)
event_ratios.columns = [f"ratio_{col}" for col in event_ratios.columns]
event_ratios = event_ratios.reset_index()

print(event_ratios.head(3))
print(f"\nShape: {event_ratios.shape}")

     user_id  ratio_comment  ratio_flag  ratio_like  ratio_share  ratio_view
0  BOT_00000            0.0         0.0    0.495934          0.0    0.504066
1  BOT_00001            0.0         0.0    0.498982          0.0    0.501018
2  BOT_00002            0.0         0.0    0.505992          0.0    0.494008

Shape: (1000, 6)


In [7]:
print(event_ratios[event_ratios["user_id"].str.startswith("USR")].head(3))

       user_id  ratio_comment  ratio_flag  ratio_like  ratio_share  ratio_view
200  USR_00000       0.204593    0.209812    0.203549     0.204593    0.177453
201  USR_00001       0.192351    0.193476    0.182227     0.204724    0.227222
202  USR_00002       0.185366    0.182927    0.193902     0.229268    0.208537


In [8]:
# Count how many users share each IP address
ip_user_count = events_df.groupby("ip_address")["user_id"].nunique().reset_index()
ip_user_count.columns = ["ip_address", "users_sharing_ip"]

# Bring it back to user level
user_ip = events_df[["user_id", "ip_address"]].drop_duplicates()
user_ip = user_ip.merge(ip_user_count, on="ip_address")

print(user_ip.head(3))
print(f"\nMax users sharing one IP: {user_ip['users_sharing_ip'].max()}")
print(f"Avg users sharing one IP: {user_ip['users_sharing_ip'].mean():.1f}")

     user_id       ip_address  users_sharing_ip
0  USR_00000   210.72.237.234                 1
1  USR_00001  215.213.125.185                 1
2  USR_00002    142.14.44.201                 1

Max users sharing one IP: 19
Avg users sharing one IP: 3.1


In [9]:
users_df.head()

,user_id,is_bot,account_age_days,ip_address,email_domain,verified,subscriber_count
0,USR_00000,0,684,210.72.237.234,hotmail.com,True,3278
1,USR_00001,0,311,215.213.125.185,gmail.com,True,29256
2,USR_00002,0,172,142.14.44.201,hotmail.com,True,88696
3,USR_00003,0,588,160.229.31.48,gmail.com,True,77397
4,USR_00004,0,462,28.214.87.139,yahoo.com,True,3905


In [12]:
# Start with users table (has our ground truth label)
features = users_df[["user_id", "is_bot", "account_age_days", 
                      "verified", "subscriber_count"]].copy()

# Merge velocity features
features = features.merge(velocity, on="user_id", how="left")

# Merge event ratio features
features = features.merge(event_ratios, on="user_id", how="left")

# Merge IP concentration
features = features.merge(user_ip[["user_id", "users_sharing_ip"]], 
                          on="user_id", how="left")

print(f"Feature table shape: {features.shape}")
print(f"\nColumns: {list(features.columns)}")
print(f"\nSample:")
print(features.head(3))

Feature table shape: (1000, 17)

Columns: ['user_id', 'is_bot', 'account_age_days', 'verified', 'subscriber_count', 'total_events', 'unique_videos', 'unique_ips', 'avg_session_duration', 'total_days_active', 'events_per_day', 'ratio_comment', 'ratio_flag', 'ratio_like', 'ratio_share', 'ratio_view', 'users_sharing_ip']

Sample:
     user_id  is_bot  account_age_days  verified  subscriber_count  \
0  USR_00000       0               684      True              3278   
1  USR_00001       0               311      True             29256   
2  USR_00002       0               172      True             88696   

   total_events  unique_videos  unique_ips  avg_session_duration  \
0           958            433           1           1815.219207   
1           889            414           1           1793.118110   
2           820            401           1           1835.128049   

   total_days_active  events_per_day  ratio_comment  ratio_flag  ratio_like  \
0                 30       31.933333  

In [13]:
features.to_csv("../data/features.csv", index=False)
print("features.csv saved ✓")
print(f"\nBot sample:")
print(features[features["is_bot"]==1][["user_id","events_per_day","users_sharing_ip","ratio_like","ratio_comment","account_age_days"]].head(3))
print(f"\nLegit sample:")
print(features[features["is_bot"]==0][["user_id","events_per_day","users_sharing_ip","ratio_like","ratio_comment","account_age_days"]].head(3))

features.csv saved ✓

Bot sample:
       user_id  events_per_day  users_sharing_ip  ratio_like  ratio_comment  \
800  BOT_00000      360.733333                13    0.495934            0.0   
801  BOT_00001      343.766667                 5    0.498982            0.0   
802  BOT_00002      358.833333                13    0.505992            0.0   

     account_age_days  
800                 3  
801                 5  
802                 3  

Legit sample:
     user_id  events_per_day  users_sharing_ip  ratio_like  ratio_comment  \
0  USR_00000       31.933333                 1    0.203549       0.204593   
1  USR_00001       29.633333                 1    0.182227       0.192351   
2  USR_00002       27.333333                 1    0.193902       0.185366   

   account_age_days  
0               684  
1               311  
2               172  
